## 1. Importação das bibliotecas necessárias

In [ ]:
import numpy as np
import json, torch, optuna
import time as time
from sklearn import preprocessing
import pickle

# importações para conectar ao código FEM
import KratosMultiphysics
from fem_interfaces.kratos.Kratos_Struct_Linear_Sudret_Truss import *

# funções relacionadas à rede neural
from neural_net.training import train_with_loader
from neural_net.data_utilities import loader_creation
from neural_net.networks import Net3
from utilities.plot_utilities import plot_data_general

## 2. Sementes para Reprodutibilidade e Dispositivo

In [ ]:
# Definindo semente para reprodutibilidade
np.random.seed(25)
torch.manual_seed(1)

In [ ]:
# Definindo o dispositivo no qual se deseja trabalhar
device = 'cuda' if torch.cuda.is_available() else 'cpu'

## 3. Geração de Dados

O código a seguir tem a finalidade de gerar os dados conforme o que está estabelecido na tabela a seguir:

| Input variable | Distribution | Mean | Standard deviation |
| -------------- | ------------ | ---- | ------------------ |
| Horizontal cross-section area $A_{h}(m^{2})$ | Normal | $1.0\times10^{-3}$ | $1.0\times10^{-4}$ |
| Vertical cross-section area $A_{v}(m^{2})$ | Normal | $2.0\times10^{-3}$ | $2.0\times10^{-4}$ |
| Horizontal Young's modulus $E_{h}(Pa)$ | Normal | $2.1\times10^{11}$ | $2.1\times10^{10}$ |
| Vertical Young's modulus $E_{v}(Pa)$ | Normal | $2.1\times10^{11}$ | $2.1\times10^{10}$ |
| Vertical forces $P1-P6(N)$ | Normal | $-5.0\times10^{5}$ | $5.0\times10^{4}$ |

In [ ]:
# Making training data of input space
n_samples = 15000

try:
    k_all   = np.load(f'data/k_all_{n_samples}.npy')
    f_all   = np.load(f'data/f_all_{n_samples}.npy')
    data_in = np.load(f'data/data_in_{n_samples}.npy')
    loaded  = True
except:
    loaded = False

if not loaded:
    E_low, E_high = 2.1e10, 2.1e12
    A_Ver_low, A_Ver_high = 1e-4, 1e-3
    A_Hor_low, A_Hor_high = 1e-4, 1e-3
    F_low, F_high = -1e4, -7.5e3

    E_Hor_rv = np.random.uniform(E_low, E_high, size=n_samples).reshape(n_samples, 1)
    E_Ver_rv = np.random.uniform(E_low, E_high, size=n_samples).reshape(n_samples, 1)
    A_Hor_rv = np.random.uniform(A_Hor_low, A_Hor_high, size=n_samples).reshape(n_samples, 1)
    A_Ver_rv = np.random.uniform(A_Ver_low, A_Ver_high, size=n_samples).reshape(n_samples, 1)
    F1_rv    = np.random.uniform(F_low, F_high, size=n_samples).reshape(n_samples, 1)
    F2_rv    = np.random.uniform(F_low, F_high, size=n_samples).reshape(n_samples, 1)
    F3_rv    = np.random.uniform(F_low, F_high, size=n_samples).reshape(n_samples, 1)
    F4_rv    = np.random.uniform(F_low, F_high, size=n_samples).reshape(n_samples, 1)
    F5_rv    = np.random.uniform(F_low, F_high, size=n_samples).reshape(n_samples, 1)
    F6_rv    = np.random.uniform(F_low, F_high, size=n_samples).reshape(n_samples, 1)

    data_in_str  = np.concatenate((E_Ver_rv, A_Ver_rv, E_Hor_rv, A_Hor_rv), axis=1)
    data_in_load = np.concatenate((F1_rv, F2_rv, F3_rv, F4_rv, F5_rv, F6_rv), axis=1)
    data_in      = np.concatenate((data_in_str, data_in_load), axis=1)
    k_all = []
    f_all = []

    # Data creation
    print("Started making data")
    with open("sim_parameters/ProjectParameters.json", 'r') as parameter_file:
        parameters = KratosMultiphysics.Parameters(parameter_file.read())

    start_time = time.time()

    # call Kratos and create matrices  here
    for data_str, data_load in zip(data_in_str, data_in_load):
        model      = KratosMultiphysics.Model()
        simulation = StructMechAnaWithVaryingParameters(model, parameters, data_str, data_load)
        k, f       = simulation.Run()
        k_all.append(k.toarray())
        f_all.append(f)

    end_time = time.time()
    print("time taken to make data is", end_time-start_time)

    k_all   = np.asarray(k_all)
    f_all   = np.asarray(f_all)
    data_in = np.asarray(data_in)

    np.save(f'data/k_all_{n_samples}', k_all)
    np.save(f'data/f_all_{n_samples}', f_all)
    np.save(f'data/data_in_{n_samples}', data_in)

## 4. Preprocessamento e Carregamento em Minibatches

In [ ]:
scaler = preprocessing.MinMaxScaler()
data_in_s = scaler.fit_transform(data_in)
pickle.dump(scaler, open('scaler.pkl', 'wb'))

# data_in_s is the scaled input data to the neural network 
data_in_torch = torch.from_numpy(data_in_s).double().to(device)

# loader to load data in batches, batch size can be given in argument 
lc = loader_creation(data_in_torch, k_all, f_all, n_samples)
train_loader, test_loader = lc.get_loaders(64, True)
feature_size = data_in.shape[1]

## 5. Determinação de Hiperparâmetros com `Optuna`

O código a seguir tem o objetivo de buscar os melhores hiperparâmetros para a rede por otimização via biblioteca `Optuna`.

In [ ]:
def objective(trial):
    # Suggest values of the hyperparameters using a trial object.
    n_layers = trial.suggest_int('n_layers', 3, 8)
    n_units  = trial.suggest_int('n_units', 32, 256)
    init     = trial.suggest_float('init', 0.1, 2)
    lr       = trial.suggest_float('lr', 1.0E-5, 1.0E-2, log=True)

    # Creation of network
    net = Net3(n_feature=feature_size, n_hidden=n_units, n_output=39, depth=n_layers, init=init).double().to(device)
    train_loss, test_loss = train_with_loader(net, train_loader, test_loader, lr, 50)

    return test_loss

# Create a study object and optimize the objective function to find the best hyperparameters.
study = optuna.create_study(direction='minimize', pruner=optuna.pruners.MedianPruner())
study.optimize(objective, n_trials=100)

In [ ]:
print(study.best_params['n_units'], study.best_params['n_layers'], study.best_params['init'], study.best_params['lr'])

In [ ]:
# Após o study.optimize
melhores_params = study.best_params

# Adicionar informações extras úteis
dados_modelo = {
    'params': melhores_params,
    'best_value': study.best_value, # A menor perda encontrada
    'feature_size': feature_size,
    'n_output': 39
}

# Salvar em arquivo
with open('hyper_params.json', 'w') as f:
    json.dump(dados_modelo, f, indent=4)

print('Parâmetros salvos com sucesso!')

## 6. Treinamento da Rede Neural

Uma vez encontrados os parâmetros por meio do estudo realizado pelo `Optuna`, a rede é treinada e, em seguida, o modelo é salvo.

In [ ]:
with open('hyper_params.json', 'r') as f:
    config = json.load(f)

params       = config['params']
feature_size = config['feature_size']
n_output     = config['n_output']

In [ ]:
net_final = Net3(n_feature=feature_size,
                 n_hidden=params['n_units'],
                 n_output=n_output,
                 depth=params['n_layers'],
                 init=params['init']).double().to(device)

train_loss, test_loss = train_with_loader(net_final, train_loader, test_loader, params['lr'], 250)

In [ ]:
# saving the model
torch.save(obj=net_final.state_dict(), f='models/model.pt')

## 7. Predição com o Modelo Substituto FEM-NN

A partir daqui, a idéia é utilizar o modelo salvo para fazer predições.

In [ ]:
# material properties
E_Hor, E_Ver = 2.1e11, 2.32e11
A_Hor, A_Ver = 9.2e-4, 1.89e-3

# forces
P1, P2, P3, P4, P5, P6 = -5.2e4, -5.2e4, -5.4e4, -3.6e4, -6.5e4, -4.4e4

data_in_str  = np.array([[E_Ver, A_Ver, E_Hor, A_Hor]])
data_in_load = np.array([[P1, P2, P3, P4, P5, P6]])
data_in      = np.concatenate((data_in_str, data_in_load), axis=1)

scalar = pickle.load(open('scaler.pkl', 'rb'))

data_in_scaled = scalar.transform(data_in)
train_in = torch.from_numpy(data_in_scaled).double().to(device)

# NN surrogate model
feature_size = data_in.shape[1]

In [ ]:
model_femnn = Net3(n_feature=feature_size,
                   n_hidden=params["n_units"],
                   n_output=n_output,
                   depth=params["n_layers"],
                   init=params["init"]).double().to(device)

model_femnn.load_state_dict(torch.load('models/model.pt'))

print('Predicting for the inputs', E_Hor, E_Ver, A_Hor, A_Ver, P1, P2, P3, P4, P5, P6)

# Making a prediction with FEM-NN surrogate model
model_femnn.eval()

with torch.inference_mode():
    u_pred = model_femnn(train_in).detach().cpu().numpy()

pred_reshaped = u_pred.reshape((13, 3))

In [ ]:
# Obtaining the FEM library solution
with open('sim_parameters/ProjectParameters.json','r') as parameter_file:
    parameters = KratosMultiphysics.Parameters(parameter_file.read())

for data_str, data_load in zip(data_in_str, data_in_load):  # replace it with all varible you wanto to change
    model_kratos = KratosMultiphysics.Model()
    simulation   = StructMechAnaWithVaryingParameters_qoi(model_kratos, parameters, data_str, data_load)
    simulation.Run()

    x_act = simulation.qoi_x
    y_act = simulation.qoi_y
    z_act = simulation.qoi_z

## 8. Plotando Soluções FEM vs. FEM-NN

In [ ]:
# Ploting both the solutions FEM and FEM_NN
x_axis = np.asarray([2*t for t in range(0, 13)])
plot_data_general(x_axis, np.asarray(y_act), pred_reshaped[:, 1], 'sudret',  ['FEM', 'FEM-NN'], ['x (m)', 'displacement (mm)'])